# Text Classification (Sentiment)

## Setup

### Import Dependencies

In [2]:
import pandas as pd
import os
import ipynbname
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
import streamlit as st

### Load CSV file into DataFrame

In [3]:
load_dotenv()

df = pd.read_csv("../data/transformed_news_data.csv")

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week
0,95883914,2026-05-22,Are AI Layoffs Really About AI? Adecco’s CEO S...,forbes.com,"('Adecco Group', 'Ai', 'Denis Machuel', 'Denni...",2026-05-22,"('aaic', 'ai', 'meta', 'meta', 'metv')",5
1,95883876,2026-05-22,Qualcomm's stock pop shows investors are 'waki...,cnbc.com,"('Breaking News: Technology', 'Business', 'Com...",2026-05-22,"('arm', 'googl', 'intc', 'meta', 'meta', 'metv...",5
2,95883673,2026-05-22,Texas Sues Mark Zuckerberg’s Meta Over Alleged...,aol.com,"('Consumer Cyclical', 'ETF', 'Ken Paxton', 'Lo...",2026-05-22,"('goog', 'googl', 'meta', 'meta', 'metv', 'nflx')",5
3,95882615,2026-05-22,Alameda Co. candidate suspended from Meta acco...,aol.com,"('Alameda County', 'Alameda County Board Of Ed...",2026-05-22,"('bsrr', 'meta', 'meta', 'metv', 'srmc', 'swir')",5
4,95881842,2026-05-22,"Unity (U), Meta (META) Extend Multi-Year VR Pl...",finance.yahoo.com,"('ETF', 'Meta', 'Platform Support', 'Stock', '...",2026-05-22,"('meta', 'meta', 'metv', 'u')",5


## Text Classification (sentiment)

### Run model on news data

In [4]:
def run_text_classification(df):
    api_variable_name = 'hugging_face_token'
    try:
        api_key = os.getenv(api_variable_name)
    except Exception as e:
        api_key = st.secrets[api_variable_name]

    # Check if user has a hugging face api key
    if api_key is None:
        raise Exception('hugging_face_token environment variable not set in a .env file')

    client = InferenceClient(
        provider="hf-inference",
        api_key=api_key,
    )

    results = client.text_classification(
        df['title'].to_list(),
        model="ProsusAI/finbert",
    )

    # Generated with AI (Gemini 3 fast)
    df['sentiment_label'] = [res.label for res in results]
    df['sentiment_score'] = [res.score for res in results]
    return df

df = run_text_classification(df)
df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week,sentiment_label,sentiment_score
0,95883914,2026-05-22,Are AI Layoffs Really About AI? Adecco’s CEO S...,forbes.com,"('Adecco Group', 'Ai', 'Denis Machuel', 'Denni...",2026-05-22,"('aaic', 'ai', 'meta', 'meta', 'metv')",5,negative,0.564054
1,95883876,2026-05-22,Qualcomm's stock pop shows investors are 'waki...,cnbc.com,"('Breaking News: Technology', 'Business', 'Com...",2026-05-22,"('arm', 'googl', 'intc', 'meta', 'meta', 'metv...",5,positive,0.723239
2,95883673,2026-05-22,Texas Sues Mark Zuckerberg’s Meta Over Alleged...,aol.com,"('Consumer Cyclical', 'ETF', 'Ken Paxton', 'Lo...",2026-05-22,"('goog', 'googl', 'meta', 'meta', 'metv', 'nflx')",5,negative,0.881771
3,95882615,2026-05-22,Alameda Co. candidate suspended from Meta acco...,aol.com,"('Alameda County', 'Alameda County Board Of Ed...",2026-05-22,"('bsrr', 'meta', 'meta', 'metv', 'srmc', 'swir')",5,negative,0.926955
4,95881842,2026-05-22,"Unity (U), Meta (META) Extend Multi-Year VR Pl...",finance.yahoo.com,"('ETF', 'Meta', 'Platform Support', 'Stock', '...",2026-05-22,"('meta', 'meta', 'metv', 'u')",5,positive,0.717177


In [5]:
def enrich_df(df):
    # Assign numerical labels to each sentiment
    df['numerical_sentiment'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive'
                                                     else (0 if x['sentiment_label'] == 'neutral' else -1),
                                                     axis=1)

    # Flag if an article is positive, negative or neutral
    df['is_positive'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive' else 0, axis=1)
    df['is_negative'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'negative' else 0, axis=1)
    df['is_neutral'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'neutral' else 0, axis=1)

    return df

df = enrich_df(df)
df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week,sentiment_label,sentiment_score,numerical_sentiment,is_positive,is_negative,is_neutral
0,95883914,2026-05-22,Are AI Layoffs Really About AI? Adecco’s CEO S...,forbes.com,"('Adecco Group', 'Ai', 'Denis Machuel', 'Denni...",2026-05-22,"('aaic', 'ai', 'meta', 'meta', 'metv')",5,negative,0.564054,-1,0,1,0
1,95883876,2026-05-22,Qualcomm's stock pop shows investors are 'waki...,cnbc.com,"('Breaking News: Technology', 'Business', 'Com...",2026-05-22,"('arm', 'googl', 'intc', 'meta', 'meta', 'metv...",5,positive,0.723239,1,1,0,0
2,95883673,2026-05-22,Texas Sues Mark Zuckerberg’s Meta Over Alleged...,aol.com,"('Consumer Cyclical', 'ETF', 'Ken Paxton', 'Lo...",2026-05-22,"('goog', 'googl', 'meta', 'meta', 'metv', 'nflx')",5,negative,0.881771,-1,0,1,0
3,95882615,2026-05-22,Alameda Co. candidate suspended from Meta acco...,aol.com,"('Alameda County', 'Alameda County Board Of Ed...",2026-05-22,"('bsrr', 'meta', 'meta', 'metv', 'srmc', 'swir')",5,negative,0.926955,-1,0,1,0
4,95881842,2026-05-22,"Unity (U), Meta (META) Extend Multi-Year VR Pl...",finance.yahoo.com,"('ETF', 'Meta', 'Platform Support', 'Stock', '...",2026-05-22,"('meta', 'meta', 'metv', 'u')",5,positive,0.717177,1,1,0,0


### Calculate Metrics

In [6]:
def calculate_metrics(df, called_externally=False):
    # Calculate metrics
    metrics_df = df.groupby(['published_date'])\
                    .agg({
                    'numerical_sentiment': 'mean',
                    'sentiment_score': 'mean',
                    'is_positive': 'sum',
                    'is_negative': 'sum',
                    'is_neutral': 'sum',
                    'sentiment_label': 'count'})\
                    .reset_index()\
                    .rename(
                    columns={'published_date': 'date',
                             'is_positive': 'positive_count',
                             'is_negative': 'negative_count',
                             'is_neutral': 'neutral_count',
                             'sentiment_label': 'total_articles',
                             'numerical': 'mean_sentiment',
                             'sentiment_score': 'mean_sentiment_probability'})

    prefix = ''
    # Allows function to be run within notebook or when called by external files
    if not called_externally:
        prefix = '../'

    file_name = f'{prefix}data/sentiment_counts.csv'

    metrics_df.to_csv(file_name, mode='a', index=False, header=None)
    sentiment_counts_df = pd.read_csv(file_name).drop_duplicates(subset=['date'], keep='first')
    sentiment_counts_df.to_csv(file_name, index=False)

    # Calculate Percentages
    metrics_df['percent_positive'] = metrics_df['positive_count'] / metrics_df['total_articles']
    metrics_df['percent_negative'] = metrics_df['negative_count'] / metrics_df['total_articles']
    metrics_df['percent_neutral'] = metrics_df['neutral_count'] / metrics_df['total_articles']

    # Remove count columns use the percentages of totals instead
    metrics_df = metrics_df.drop(['positive_count', 'negative_count', 'neutral_count', 'total_articles'], axis=1)

    return metrics_df

metrics_df = calculate_metrics(df)
metrics_df.head()

,date,numerical_sentiment,mean_sentiment_probability,percent_positive,percent_negative,percent_neutral
0,2026-05-20,-0.350877,0.868921,0.105263,0.456140,0.438596
1,2026-05-21,-0.225352,0.787078,0.140845,0.366197,0.492958
2,2026-05-22,-0.159091,0.820265,0.204545,0.363636,0.431818


## Write output DataFrame as CSV File

In [7]:
# Output as CSV
metrics_df.to_csv('../data/article_metrics.csv', index=False)